<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B07%5D%20-%20Ingenieria_de_Variables_I/%5B01%5D%20-%20Notebooks/E4_Caza_del_Leakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E4 · Caza del leakage — Ingenieria de Variables I (bonus)

## Introduccion

> **Leakage (fuga de datos)**: el modelo usa informacion que **no tendra** en el momento
> real de predecir. Score buenisimo en pruebas, desastre en produccion.
>
> En cristiano: es aprobar un examen porque alguien te paso las soluciones la noche antes.
> En clase pareces brillante; el dia real, sin chuleta, te hundes.

Regla de oro: pregunta siempre **"¿esta informacion existe cuando voy a predecir?"**.
Y recuerda: un **acierto casi perfecto** es para encender la alarma.

En este notebook hay **3 trozos de codigo con fuga**. Cada caso aparece primero **mal**
(con la fuga) y luego **arreglado**:

1. `fit` (escalado) **antes** del split.
2. Una **feature del futuro**.
3. **Fuga temporal** (split aleatorio en datos con tiempo).

## Objetivos del ejercicio

- Reconocer las tres fugas mas frecuentes y por que inflan las metricas.
- Arreglar cada una con la practica correcta (split primero, eliminar features del futuro, split temporal).
- Repasar **los 6 tropiezos clasicos** de la ingenieria de variables.

### 1. Importar librerias necesarias

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

### 2. Datos con dimension temporal

Generamos transacciones **ordenadas por fecha** y con una **deriva (drift)**: la tasa de
fraude crece a lo largo del año. Tambien añadimos una columna trampa, `revision_posterior`,
que solo se conoce **despues** de investigar (informacion del futuro).

In [ ]:
def generar_datos_temporales(n=8000, semilla=7):
    rng = np.random.default_rng(semilla)
    inicio = np.datetime64("2024-01-01T00:00")
    minutos = np.sort(rng.integers(0, 365 * 24 * 60, size=n))   # ordenado por tiempo
    fecha = pd.to_datetime(inicio + minutos.astype("timedelta64[m]"))

    t = np.linspace(0, 1, n)                       # progreso temporal 0..1
    monto = np.round(rng.lognormal(mean=3.0, sigma=0.9, size=n), 2)
    hora = fecha.hour.to_numpy()
    online = rng.choice([0, 1], size=n, p=[0.55, 0.45])

    # DRIFT: ademas de subir la tasa, CAMBIA la relacion variable-target con el tiempo.
    # Al principio pesa la madrugada; al final pesa mucho mas el canal online.
    logit = (-4.0 + 1.0 * t
             + 0.5 * (np.log1p(monto) - np.log1p(monto).mean())
             + (2.5 - 2.0 * t) * (hora < 6)        # la madrugada importa al principio, luego menos
             + (0.2 + 3.0 * t) * online)           # online empieza flojo y acaba siendo clave
    prob = 1.0 / (1.0 + np.exp(-logit))
    es_fraude = rng.binomial(1, prob)

    # Columna TRAMPA: resultado de la revision posterior (solo existe en el futuro)
    revision_posterior = np.where(rng.random(n) < 0.92, es_fraude, 1 - es_fraude)

    return pd.DataFrame({
        "fecha": fecha, "monto": monto, "log_monto": np.log1p(monto),
        "hora": hora, "online": online,
        "revision_posterior": revision_posterior, "es_fraude": es_fraude,
    })

datos = generar_datos_temporales()
print("Forma:", datos.shape, "| Tasa de fraude global:", round(datos["es_fraude"].mean(), 3))
print("Fraude en el primer 20% del tiempo:", round(datos.head(int(len(datos)*0.2))["es_fraude"].mean(), 3))
print("Fraude en el ultimo 20% del tiempo:", round(datos.tail(int(len(datos)*0.2))["es_fraude"].mean(), 3))
datos.head()

### Caso 1 · `fit` (escalado) ANTES del split  ❌

Si ajustamos el `StandardScaler` con **todos** los datos, el conjunto de test influye en la
media/desviacion que usamos: el test se "cuela" en el preprocesado.

In [ ]:
features = ["monto", "log_monto", "hora", "online"]
X = datos[features].copy()
y = datos["es_fraude"]

# ❌ MAL: escalar con TODO antes de separar
scaler_mal = StandardScaler().fit(X)                 # <-- ve train + test
X_escalado = scaler_mal.transform(X)
Xtr, Xte, ytr, yte = train_test_split(X_escalado, y, test_size=0.3, random_state=0, stratify=y)
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"[Caso 1 - MAL] AUC test: {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}")

In [ ]:
# ✅ BIEN: separar primero y ajustar el scaler SOLO con train
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
scaler_ok = StandardScaler().fit(Xtr)                # <-- solo train
Xtr_s = scaler_ok.transform(Xtr)
Xte_s = scaler_ok.transform(Xte)
m = LogisticRegression(max_iter=1000).fit(Xtr_s, ytr)
print(f"[Caso 1 - BIEN] AUC test: {roc_auc_score(yte, m.predict_proba(Xte_s)[:,1]):.4f}")
print("\nCon el escalado el efecto es pequeño, pero la PRACTICA es incorrecta: con imputacion")
print("o target encoding la fuga puede ser enorme. Solucion definitiva: meterlo en un Pipeline.")

### Caso 2 · Una feature del futuro  ❌

`revision_posterior` es el resultado de investigar la transaccion: **no existe** cuando hay
que predecir. Si la metemos como feature, el AUC se dispara a casi 1.0 → **señal de alarma**.

In [ ]:
# ❌ MAL: incluir una variable que viene del futuro
feat_mal = ["monto", "log_monto", "hora", "online", "revision_posterior"]
Xtr, Xte, ytr, yte = train_test_split(datos[feat_mal], y, test_size=0.3, random_state=0, stratify=y)
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"[Caso 2 - MAL] AUC test: {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}  <- demasiado bueno")

In [ ]:
# ✅ BIEN: eliminar la feature del futuro
feat_ok = ["monto", "log_monto", "hora", "online"]
Xtr, Xte, ytr, yte = train_test_split(datos[feat_ok], y, test_size=0.3, random_state=0, stratify=y)
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"[Caso 2 - BIEN] AUC test: {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}  <- realista")
print("\nRegla: si una variable solo se conoce DESPUES del evento, no puede ser feature.")

### Caso 3 · Fuga temporal (split aleatorio en datos con tiempo)  ❌

Con **drift**, un split **aleatorio** mezcla pasado y futuro: el modelo entrena con filas del
futuro y la estimacion sale optimista. En produccion siempre predecimos el **futuro** a partir
del **pasado**, asi que la validacion debe ser **temporal**.

In [ ]:
# ❌ MAL: split aleatorio (mezcla pasado y futuro)
Xtr, Xte, ytr, yte = train_test_split(datos[feat_ok], y, test_size=0.3, random_state=0, shuffle=True)
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"[Caso 3 - MAL] AUC (split aleatorio): {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}")

In [ ]:
# ✅ BIEN: split temporal (entrenar con el pasado, evaluar con el futuro)
corte = int(len(datos) * 0.7)
train_t = datos.iloc[:corte]      # primer 70% en el tiempo
test_t = datos.iloc[corte:]       # ultimo 30% en el tiempo
m = LogisticRegression(max_iter=1000).fit(train_t[feat_ok], train_t["es_fraude"])
auc_temporal = roc_auc_score(test_t["es_fraude"], m.predict_proba(test_t[feat_ok])[:, 1])
print(f"[Caso 3 - BIEN] AUC (split temporal): {auc_temporal:.4f}")
print("\nEl split temporal suele dar un AUC mas bajo... pero es el que veras en produccion.")

### Los 6 tropiezos clasicos (resumen)

1. **Leakage**: usar info que no existe al predecir.
2. **Transformar antes de separar** train y test (fit en todo el dataset).
3. **Target encoding sin turnos ni smoothing**.
4. **Categorias nuevas en test** que no estaban en train (usar `handle_unknown="ignore"`).
5. **Drift** entre train y produccion (validar en el tiempo, vigilar con PSI).
6. **Pseudo-numericas** (un codigo postal tratado como numero).

### Reflexion

1. ¿Por que el escalado antes del split apenas mueve el AUC pero la imputacion o el target
   encoding antes del split pueden moverlo muchisimo?
2. ¿Que pregunta unica te protege de casi todas las fugas?
3. ¿Por que un AUC de 0.99 deberia darte miedo en lugar de alegria?
4. ¿Cuando es obligatorio un split temporal en vez de uno aleatorio?